# B2 — L'influence des influents

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| B — Influence dans un réseau social | 2 sur 4 | 3 h | M1, M4, M7 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le degré compte les amis d'un membre, sans regarder **qui** ils sont. Or, dans un club, avoir l'oreille de deux personnes écoutées par tous compte souvent plus que dix amis isolés. On aimerait une définition **récursive** :

> un membre est influent s'il est lié à des membres influents.

Cette définition semble tourner en rond. Le module M7 a montré comment en sortir : c'est une équation de vecteur propre.

**La sous-question de cette étape : comment calculer une influence récursive, quelles variantes existe-t-il, et à quel point le classement dépend-il des choix de calcul et des données ?**

**Activité** : Dans le réseau du club, citez un membre de faible degré qui pourrait pourtant être influent au sens récursif, en regardant le dessin de l'étape B1. Expliquez pourquoi.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_premier_vecteur_propre = 34      # membre classé premier par la centralité de vecteur propre (modifiez)
pari_classements_tres_differents = "non"   # le classement récursif sera-t-il très différent du degré ? (modifiez)

## ③ Abstraire : trois influences récursives

![Schéma : trois façons de mesurer l'influence ; le degré compte les liens, la centralité de vecteur propre tient compte de l'importance des voisins, la centralité de Katz compte tous les chemins avec un poids qui décroît avec leur longueur](../../../assets/schemas/schema-centralites.svg)

**La centralité de vecteur propre.** On demande que l'influence $x_i$ de chaque membre soit proportionnelle à la somme des influences de ses voisins :

$$
\lambda\,x_i = \sum_j A_{ij}\,x_j \qquad\Longleftrightarrow\qquad A\mathbf{x} = \lambda\,\mathbf{x} .
$$

Pour un réseau connexe, le théorème de Perron-Frobenius (module M7) garantit qu'il existe un seul vecteur propre à coefficients tous positifs : celui de la plus grande valeur propre $\lambda_1$. La méthode de la puissance le calcule.

**La centralité de Katz.** On compte **tous** les chemins qui partent d'un membre, en donnant aux chemins de longueur $k$ le poids $\alpha^k$ (avec $0 < \alpha < 1$) :

$$
\mathbf{x} = \alpha A\mathbf{1} + \alpha^2 A^2\mathbf{1} + \alpha^3 A^3\mathbf{1} + \cdots = \left((I - \alpha A)^{-1} - I\right)\mathbf{1} .
$$

La somme ne converge que si $\alpha < 1/\lambda_1$ : c'est alors que $I - \alpha A$ est inversible et que la série géométrique de matrices a un sens (modules M4 et M7). Pour $\alpha$ très petit, seuls les chemins de longueur 1 comptent : on retrouve le degré.

**Le PageRank** (projet du module M7) : un surfeur aléatoire suit les liens, avec des sauts au hasard. Sa matrice de transition divise chaque colonne de $A$ par le degré.

**Activité** : Pour le réseau à cinq membres du schéma de l'étape B1, partez de $\mathbf{x}_0 = \mathbf{1}$ et calculez à la main $A\mathbf{x}_0$ puis $A^2\mathbf{x}_0$. Quel membre gagne le plus en importance entre les deux étapes ? Pourquoi ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### La centralité de vecteur propre

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reseaux import charger, dessiner

membres, liens, A = charger("karate")
_, _, A_poids = charger("karate", pondere=True)
n = len(A)

def vecteur_propre(A):
    valeurs, vecteurs = np.linalg.eigh(A)          # A est symétrique : eigh renvoie des réels, valeurs croissantes
    v = np.abs(vecteurs[:, -1])                    # vecteur de la plus grande valeur propre, signe rendu positif
    return valeurs[-1], v / v.sum()

def puissance(A, iterations=200):
    x = np.ones(len(A))
    for _ in range(iterations):
        x = A @ x
        x = x / x.sum()
    return x

lam1, propre = vecteur_propre(A)
print(f"λ1 = {lam1:.4f} ; méthode de la puissance identique ? {np.allclose(puissance(A), propre)}")

mesures = pd.DataFrame({"membre": membres["id"], "rôle": membres["role"].fillna(""), "degré": A.sum(axis=1).astype(int),
                        "vecteur propre": propre})
print(mesures.sort_values("vecteur propre", ascending=False).head(8).round(4).to_string(index=False))

### La centralité de Katz, et son réglage $\alpha$

In [ ]:
def katz(A, alpha):
    n = len(A)
    return np.linalg.solve(np.eye(n) - alpha * A, np.ones(n)) - 1

alphas = np.linspace(0.002, 0.98, 60) / lam1          # de presque 0 à presque 1/λ1
accord_degre = [pd.Series(katz(A, a)).rank().corr(mesures["degré"].rank()) for a in alphas]
accord_propre = [pd.Series(katz(A, a)).rank().corr(mesures["vecteur propre"].rank()) for a in alphas]
conditionnement = [np.linalg.cond(np.eye(n) - a * A) for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(alphas * lam1, accord_degre, color=LIOUAI["navy"], label="avec le degré")
axes[0].plot(alphas * lam1, accord_propre, color=LIOUAI["vert"], label="avec le vecteur propre")
axes[0].set_xlabel("α × λ1"); axes[0].set_ylabel("corrélation des rangs"); axes[0].legend(); axes[0].set_title("Katz entre deux extrêmes")
axes[1].semilogy(alphas * lam1, conditionnement, color=LIOUAI["navy"])
axes[1].set_xlabel("α × λ1"); axes[1].set_ylabel("cond(I − αA)"); axes[1].set_title("Le calcul devient instable près de 1/λ1")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

trop_grand = katz(A, 1.2 / lam1)
print(f"Avec α = 1,2/λ1 : {np.sum(trop_grand < 0)} membres ont une « influence » négative : le calcul n'a plus de sens.")

**Activité** : Décrivez le graphique de gauche. Que devient la centralité de Katz quand $\alpha$ tend vers 0 ? vers $1/\lambda_1$ ? Pourquoi le calcul devient-il instable à droite (module M4) ?

Indiquez vos réponses et justifiez-les.

### Le PageRank

In [ ]:
def pagerank(A, d=0.85, iterations=200):
    n = len(A)
    M = A / A.sum(axis=0)                       # colonnes de somme 1 (aucun membre isolé ici)
    r = np.ones(n) / n
    for _ in range(iterations):
        r = d * M @ r + (1 - d) / n
    return r

mesures["Katz (α = 0,5/λ1)"] = katz(A, 0.5 / lam1)
mesures["PageRank"] = pagerank(A)
mesures["vecteur propre (pondéré)"] = vecteur_propre(A_poids)[1]

colonnes = ["degré", "vecteur propre", "Katz (α = 0,5/λ1)", "PageRank", "vecteur propre (pondéré)"]
podiums = pd.DataFrame({c: mesures.sort_values(c, ascending=False)["membre"].head(6).to_list() for c in colonnes},
                       index=[f"{k}e" for k in range(1, 7)])
print(podiums.to_string())
print("\nCorrélation des rangs :")
print(mesures[colonnes].rank().corr().round(2).to_string())

### Robustesse : et si un lien était faux ?

Les données comptent 77 ou 78 liens selon les versions. On retire chaque lien, un par un, et l'on regarde si le classement change.

In [ ]:
i_liens, j_liens = np.nonzero(np.triu(A))
premiers, top5_identiques, ecarts = [], 0, []
top5_ref = set(np.argsort(-propre)[:5])
for i, j in zip(i_liens, j_liens):
    B = A.copy()
    B[i, j] = B[j, i] = 0
    _, p = vecteur_propre(B)
    premiers.append(int(membres["id"][np.argmax(p)]))
    top5_identiques += set(np.argsort(-p)[:5]) == top5_ref
    ecarts.append(np.abs(p - propre).max())
print(f"Sur {len(i_liens)} retraits d'un lien :")
print("  premier du classement :", pd.Series(premiers).value_counts().to_dict())
print(f"  les 5 premiers restent les mêmes dans {top5_identiques} cas")
print(f"  plus grand changement de centralité : {max(ecarts):.4f} (centralité maximale : {propre.max():.4f})")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : premier membre {pari_premier_vecteur_propre} ; classements très différents : {pari_classements_tres_differents}")
print(f"Résultat   : premier membre {mesures.loc[mesures['vecteur propre'].idxmax(), 'membre']}")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
dessiner(A, membres, mesures["degré"], ax=axes[0], titre="degré")
dessiner(A, membres, mesures["vecteur propre"], ax=axes[1], titre="centralité de vecteur propre")
plt.tight_layout(); plt.show()

**Activité** :

1. Quels membres gagnent ou perdent des places entre le degré et la centralité de vecteur propre ? Expliquez-le avec leur position dans le réseau.
2. Les cinq mesures désignent-elles les mêmes « influents » ? Sur quoi s'accordent-elles, sur quoi divergent-elles ?
3. Pondérer les liens change-t-il la conclusion ? Que mesure alors l'influence ?
4. Le classement résiste-t-il à l'erreur d'un lien ? Qu'en concluez-vous sur la fiabilité de vos conclusions ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le choix de l'outil.** Chaque mesure répond à une question différente : laquelle ? Pour diffuser une information au plus grand nombre, pour convaincre les membres « qui comptent », pour être vu par un visiteur qui erre de lien en lien…
- **Un réglage caché.** Le facteur $d$ du PageRank et le facteur $\alpha$ de Katz sont des constantes choisies par l'analyste. Comment justifier une valeur ?
- **Généraliser : Katz sans inverser.** La série $\mathbf{x} = \alpha A(\mathbf{1} + \mathbf{x})$ suggère un calcul par itérations : $\mathbf{x}_{k+1} = \alpha A(\mathbf{1} + \mathbf{x}_k)$, en partant de $\mathbf{x}_0 = \mathbf{0}$. Écrivez une fonction `katz_iteratif(A, alpha, tol)` qui renvoie la centralité et le nombre d'itérations. Comparez avec `katz` pour $\alpha = 0{,}5/\lambda_1$ et $\alpha = 0{,}99/\lambda_1$. Que se passe-t-il pour $\alpha > 1/\lambda_1$ ?

In [ ]:
def katz_iteratif(A, alpha, tol=1e-10, max_iter=100_000):
    # À compléter : renvoie (x, nombre d'itérations)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def katz_iteratif(A, alpha, tol=1e-10, max_iter=100_000):
    x = np.zeros(len(A))
    for k in range(1, max_iter + 1):
        with np.errstate(over="ignore", invalid="ignore"):
            nouveau = alpha * A @ (1 + x)
        if not np.isfinite(nouveau).all():
            raise OverflowError(f"Divergence après {k} itérations : α est trop grand.")
        if np.abs(nouveau - x).max() < tol:
            return nouveau, k
        x = nouveau
    raise RuntimeError("Pas de convergence")

for facteur in (0.5, 0.99):
    x, k = katz_iteratif(A, facteur / lam1)
    print(f"α = {facteur}/λ1 : {k:5d} itérations, écart avec la résolution directe {np.abs(x - katz(A, facteur / lam1)).max():.1e}")
try:
    katz_iteratif(A, 1.1 / lam1)
except (OverflowError, RuntimeError) as erreur:
    print("α = 1,1/λ1 :", erreur)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Transformer des constantes en paramètres": 1,
    "Comparer les mérites des outils dans le contexte": 1,
    "Repérer les traits intéressants du résultat": 1,
    "Quantifier la validité et l'impact des outils et concepts choisis": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "B2-influence-recursive", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")